In [1]:
# ==============================
# SQL QUERY GENERATOR (Single Script)
# Natural Language to SQL
# ==============================

import re
import sqlite3
import gradio as gr
from datetime import datetime

# ------------------------------
# Database Schema Definition
# ------------------------------
schema = {
    "users": ["id", "name", "email", "signup_date", "age", "country", "status"],
    "orders": ["id", "user_id", "product_name", "amount", "order_date", "status"],
    "products": ["id", "name", "price", "category", "stock"],
    "transactions": ["id", "user_id", "amount", "date", "type"]
}

# ------------------------------
# Security Validation
# ------------------------------
dangerous_keywords = [
    "DROP", "DELETE", "ALTER", "TRUNCATE",
    "INSERT", "UPDATE", "EXEC", "EXECUTE"
]

dangerous_patterns = [";", "--", "/*", "*/"]

def is_safe(text):
    upper = text.upper()
    for keyword in dangerous_keywords:
        if keyword in upper:
            return False
    for pattern in dangerous_patterns:
        if pattern in text:
            return False
    return True


# ------------------------------
# Table Detection
# ------------------------------
table_keywords = {
    "users": ["user", "users", "customer", "account", "profile"],
    "orders": ["order", "orders", "purchase", "buy"],
    "products": ["product", "products", "item", "goods"],
    "transactions": ["transaction", "transactions", "payment"]
}

def get_table_name(query):
    query = query.lower()
    for table, keywords in table_keywords.items():
        for k in keywords:
            if k in query:
                return table
    return None


# ------------------------------
# Column Detection
# ------------------------------
column_keywords = {
    "name": ["name", "username"],
    "email": ["email", "mail"],
    "signup_date": ["signup", "joined", "registered"],
    "age": ["age", "years"],
    "country": ["country", "location"],
    "status": ["status"],
    "amount": ["amount", "price", "total"],
}

def get_columns(query, table):
    query = query.lower()
    columns = []

    for column, keywords in column_keywords.items():
        for k in keywords:
            if k in query and column in schema.get(table, []):
                columns.append(column)

    if not columns:
        columns = ["*"]

    return columns


# ------------------------------
# Condition Extraction
# ------------------------------
def extract_conditions(query):

    query = query.lower()
    conditions = []

    amount_match = re.search(r'greater than (\d+)', query)
    if amount_match:
        value = amount_match.group(1)
        conditions.append(f"amount > {value}")

    if "last week" in query:
        conditions.append("DATE(date) >= DATE('now','-7 days')")

    if "last month" in query:
        conditions.append("DATE(signup_date) >= DATE('now','-1 month')")

    return conditions


# ------------------------------
# SQL Generator
# ------------------------------
def generate_sql(query):

    if not is_safe(query):
        return "⚠️ Unsafe query detected. Query blocked."

    table = get_table_name(query)

    if not table:
        return "❌ Could not detect table."

    columns = get_columns(query, table)

    conditions = extract_conditions(query)

    sql = f"SELECT {', '.join(columns)} FROM {table}"

    if conditions:
        sql += " WHERE " + " AND ".join(conditions)

    sql += " LIMIT 10"

    return sql


# ------------------------------
# Query History
# ------------------------------
history = []

def process_query(user_input):

    sql = generate_sql(user_input)

    history.append((user_input, sql))

    return sql


# ------------------------------
# Export to Text
# ------------------------------
def export_history():

    filename = f"sql_queries_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

    with open(filename, "w") as f:
        for q, s in history:
            f.write(f"User Query: {q}\n")
            f.write(f"SQL: {s}\n\n")

    return filename


# ------------------------------
# Gradio Interface
# ------------------------------
with gr.Blocks(title="SQL Query Generator") as demo:

    gr.Markdown("# AI SQL Query Generator")
    gr.Markdown("Convert Natural Language → SQL")

    user_input = gr.Textbox(label="Enter your query")

    output = gr.Code(label="Generated SQL", language="sql")

    btn = gr.Button("Generate SQL")

    export_btn = gr.Button("Export History")

    file_output = gr.File()

    btn.click(process_query, inputs=user_input, outputs=output)

    export_btn.click(export_history, outputs=file_output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://67090802d2727f63aa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
